<a href="https://colab.research.google.com/github/luisfelipetrj99/ai-data-science-portfolio/blob/main/enterprise-text-to-sql-rag-agent/enterprise-text-to-sql-rag-agent_english_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Enterprise Text-to-SQL RAG Agent

## Project Overview

This project implements an enterprise-grade **Text-to-SQL AI Agent** capable of translating natural language business questions into executable SQL queries over a corporate relational database.

The solution combines:
- Large Language Models (LLMs)
- Retrieval-Augmented Generation (RAG)
- Embeddings and semantic search
- FAISS vector indexing
- Metadata-driven schema retrieval
- Prompt engineering
- SQL generation and validation

The project simulates a realistic enterprise analytics environment where users from business teams can ask questions in natural language and automatically obtain SQL queries to retrieve insights from structured databases.

---

## Dataset Description

The dataset used for this challenge contains:

### Database Metadata
For every table in the database:
- Table schema
- Column definitions
- Table descriptions
- Business context

### Query Examples
Each table also includes:
- Example SQL queries
- Natural language descriptions of the queries
- Example business use cases

This metadata is used as contextual knowledge for the Text-to-SQL agent.

---

## Main Objectives

The objectives of this project are:

1. Build a corporate Text-to-SQL assistant
2. Reduce hallucinations using RAG
3. Improve SQL generation quality through semantic retrieval
4. Optimize token consumption compared to naive prompting
5. Create a scalable architecture for enterprise AI analytics assistants

---

## Architecture

The solution includes multiple approaches:

### 1. Baseline Text-to-SQL
A direct prompting strategy where the full schema context is injected into the LLM.

### 2. RAG-Based Text-to-SQL
A retrieval-based architecture where only relevant schema and examples are retrieved using embeddings and FAISS similarity search.

### 3. Key-Value Metadata Retrieval
Additional metadata optimization strategy for retrieving contextual information at table level.

---

## Topics Covered

This notebook covers:

- Natural Language Processing (NLP)
- Text-to-SQL systems
- Retrieval-Augmented Generation (RAG)
- Vector databases
- Semantic search
- Prompt engineering
- Embeddings
- FAISS indexing
- LLM orchestration
- Metadata retrieval
- SQL generation
- Token optimization
- Enterprise AI systems
- AI Agents
- Information retrieval

---

## Skills Demonstrated

### Machine Learning & AI
- NLP systems development
- LLM application development
- Embedding-based retrieval
- AI agent design

### Data Engineering
- SQL generation pipelines
- Metadata processing
- Schema retrieval systems

### MLOps & AI Engineering
- Modular AI architecture
- Retrieval pipelines
- Scalable prompt design

### Software Engineering
- Python development
- Reusable pipeline design
- Enterprise-ready architecture

---

## Potential Business Applications

This type of solution can be applied to:

- Self-service analytics
- BI copilots
- Enterprise data assistants
- Customer support analytics
- CRM analytics
- Financial reporting assistants
- Operations monitoring
- Marketing analytics

---

## Future Improvements

Potential improvements include:

- SQL validation guardrails
- Query execution feedback loops
- Multi-agent orchestration
- Query optimization
- Fine-tuning on enterprise SQL data
- Conversational memory
- Role-based access control
- Dashboard generation
- Streaming analytics integration

---

## Technologies Used

- Python
- OpenAI API
- FAISS
- Pandas
- NumPy
- Jupyter Notebook
- Embeddings
- RAG pipelines

---
## Acknowledgment

The dataset and the original project idea belong to Colegio de Matemáticas Bourbaki and were developed as part of the Advanced NLP course.

This repository presents a personal implementation and extension of the project for academic, learning, and professional portfolio purposes.


In [ ]:
!pip install faiss-cpu

In [ ]:
!pip -q install openai langchain-core

In [ ]:
import pandas as pd
import getpass
from openai import OpenAI
import numpy as np

#api_key = getpass.getpass("Introduce tu OpenAI API Key: ")
#client = OpenAI(api_key=api_key)

url = "https://raw.githubusercontent.com/luisfelipetrj99/ai-data-science-portfolio/refs/heads/main/projects/enterprise-text-to-sql-rag-agent/data/sql_dataset_bourbaki.json"

df = pd.read_json(url)

# First rows
print(df.head())

# General info
print(df.info())

                                                         users  \
description  Almacena información de cuentas de clientes, i...   
schema       id INT PRIMARY KEY, first_name VARCHAR(50), la...   
examples     [{'description': 'Buscar un usuario por correo...   

                                                    categories  \
description  Categorías de productos para organizar el catá...   
schema       id INT PRIMARY KEY, parent_id INT, name VARCHA...   
examples     [{'description': 'Obtener todas las categorías...   

                                                      products  \
description  Detalles sobre artículos a la venta, precios, ...   
schema       id INT PRIMARY KEY, category_id INT, name VARC...   
examples     [{'description': 'Listar productos inactivos o...   

                                                        orders  \
description  Datos transaccionales de alto nivel para compr...   
schema       id INT PRIMARY KEY, user_id INT, order_date TI...   
example

In [ ]:
df['users']

,users
description,"Almacena información de cuentas de clientes, i..."
schema,"id INT PRIMARY KEY, first_name VARCHAR(50), la..."
examples,[{'description': 'Buscar un usuario por correo...


In [ ]:
# Using the users table as an example to inspect its contents
for i, row in enumerate(df['users']):
    print(f"\n--- Registro {i} ---\n")
    print(row)


--- Registro 0 ---

Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.

--- Registro 1 ---

id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP

--- Registro 2 ---

[{'description': 'Buscar un usuario por correo electrónico.', 'sql': "SELECT * FROM users WHERE email = 'cliente@example.com';"}, {'description': 'Contar nuevos usuarios en el último mes.', 'sql': "SELECT COUNT(*) FROM users WHERE created_at >= CURRENT_DATE - INTERVAL '1 month';"}, {'description': 'Buscar usuarios sin número de teléfono.', 'sql': 'SELECT id, first_name, last_name FROM users WHERE phone IS NULL;'}]


## Model Baseline:

To establish a baseline for this model, each table will be converted into its own document, including the information provided for each table in the JSON file. This data will then be used to build the first version of the model.

In [ ]:
from langchain_core.documents import Document
GENERATION_MODEL="gpt-4.1-mini"

In [ ]:
def build_table_document(table_name, description, schema, examples):
    """
    Builds a structured text document containing metadata and SQL examples
    for a database table.

    This function formats the table information into a single text block
    that can later be used for embedding generation, semantic retrieval,
    Retrieval-Augmented Generation (RAG), or LLM context injection.

    Parameters
    ----------
    table_name : str
        Name of the database table.

    description : str
        Business or technical description of the table.

    schema : str
        Schema definition of the table, including columns and data types.

    examples : list of dict
        List containing example SQL queries and their descriptions.
        Each dictionary is expected to contain:
            - "description": Natural language explanation of the query.
            - "query": SQL query example.

    Returns
    -------
    str
        A formatted text document containing:
            - Table name
            - Table description
            - Table schema
            - Example SQL queries with descriptions
    """

    examples_text = ""

    for i, ex in enumerate(examples, 1):
        examples_text += f"\n{i}. Description: {ex.get('description')}\n"
        examples_text += f"   SQL: {ex.get('query')}\n"

    doc_text = f"""
Table Name: {table_name}

Description:
{description}

Schema:
{schema}

Examples:
{examples_text}
"""

    return doc_text.strip()

In [ ]:
# Initialize an empty list to store all generated documents
documents = []

# Iterate through each table available in the dataframe
for table in df.columns:

    # Extract the table description from the dataframe
    description = df.loc["description", table]

    # Extract the schema information for the current table
    schema = df.loc["schema", table]

    # Extract example SQL queries and their descriptions
    examples = df.loc["examples", table]

    # Build a structured text representation of the table
    # including metadata, schema, and SQL examples
    content = build_table_document(
        table_name=table,
        description=description,
        schema=schema,
        examples=examples
    )

    # Create a LangChain Document object
    # page_content contains the textual representation
    # metadata stores additional searchable information
    doc = Document(
        page_content=content,
        metadata={
            "table_name": table,
            "doc_type": "table_full"
        }
    )

    # Add the generated document to the documents collection
    documents.append(doc)

In [ ]:
documents

[Document(metadata={'table_name': 'users', 'doc_type': 'table_full'}, page_content='Table Name: users\n\nDescription:\nAlmacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.\n\nSchema:\nid INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP\n\nExamples:\n\n1. Description: Buscar un usuario por correo electrónico.\n   SQL: None\n\n2. Description: Contar nuevos usuarios en el último mes.\n   SQL: None\n\n3. Description: Buscar usuarios sin número de teléfono.\n   SQL: None'),
 Document(metadata={'table_name': 'categories', 'doc_type': 'table_full'}, page_content='Table Name: categories\n\nDescription:\nCategorías de productos para organizar el catálogo jerárquicamente.\n\nSchema:\nid INT PRIMARY KEY, parent_id INT, name VARCHAR(100), description TEXT\n\nExamples:\n\n1. Description: Obtener todas las categorías principales (sin padre).\n   SQL

In [ ]:
#Promp generated in OpenAI
SYSTEM_PROMPT = (
    "Eres un asistente experto en generación de consultas SQL.\n\n"

    "Tu tarea es generar consultas SQL basándote únicamente en el contexto proporcionado.\n"
    "El contexto incluye:\n"
    "- Nombre de las tablas\n"
    "- Descripción de las tablas\n"
    "- Schema (columnas y tipos)\n"
    "- Ejemplos de queries\n\n"

    "Reglas estrictas:\n"
    "1. SOLO puedes usar las tablas mencionadas en el contexto.\n"
    "2. SOLO puedes usar las columnas definidas en el schema.\n"
    "3. NO inventes tablas, columnas ni relaciones.\n"
    "4. Si la información no es suficiente para responder, di: 'No hay suficiente información para generar la consulta'.\n"
    "5. Genera únicamente consultas SQL válidas.\n"
    "6. NO incluyas explicaciones, comentarios ni texto adicional.\n"
    "7. NO uses lenguaje natural, SOLO SQL.\n\n"

    "Buenas prácticas:\n"
    "- Usa alias cuando sea necesario.\n"
    "- Usa JOINs correctamente si la consulta requiere múltiples tablas.\n"
    "- Usa funciones agregadas (COUNT, SUM, AVG, etc.) cuando aplique.\n"
    "- Aplica filtros (WHERE) de forma correcta.\n\n"

    "Formato de salida:\n"
    "Devuelve únicamente la consulta SQL en texto plano.\n"
)

In [ ]:
import os
import getpass

from openai import OpenAI
from google.colab import userdata
api_key = getpass.getpass("Introduce an OpenAI API Key: ")
client=OpenAI(api_key=api_key)


Introduce tu OpenAI API Key: ··········


In [ ]:
#Concatenated text
contexto_completo = "\n\n" + ("\n\n" + "="*80 + "\n\n").join(
    [doc.page_content for doc in documents]
)

print("Text Lenght:", len(contexto_completo))
print("\nContext Preview:\n")
print(contexto_completo[:2000])

Longitud del contexto: 4440

Vista previa del contexto:



Table Name: users

Description:
Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.

Schema:
id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP

Examples:

1. Description: Buscar un usuario por correo electrónico.
   SQL: None

2. Description: Contar nuevos usuarios en el último mes.
   SQL: None

3. Description: Buscar usuarios sin número de teléfono.
   SQL: None


Table Name: categories

Description:
Categorías de productos para organizar el catálogo jerárquicamente.

Schema:
id INT PRIMARY KEY, parent_id INT, name VARCHAR(100), description TEXT

Examples:

1. Description: Obtener todas las categorías principales (sin padre).
   SQL: None

2. Description: Listar todas las subcategorías para el ID de categoría 5.
   SQL: None


Table Name: products

Description:
Detalles so

In [ ]:
#Example QUestion
pregunta= "Dame todos los usuarios registrados con su correo electrónico."
response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Contexto:\n{contexto_completo}\n\nPregunta: {pregunta}"
            }
        ],
        temperature=0
    )


In [ ]:
sql_generado = response.choices[0].message.content

print(sql_generado)

SELECT id, first_name, last_name, email FROM users;


For a relatively simple query without filters or complex conditions, the baseline approach appears to perform correctly. In this case, the model successfully identified the appropriate table and selected the required fields, including the email information requested for all customers.

The generated SQL query was:

SELECT id, first_name, last_name, email FROM users

Next, the model will be evaluated using:
1. A question that should not be answerable based on the available database context.
2. A more complex business query to assess the model's capability to generate more advanced SQL statements involving additional logic and reasoning.

In [ ]:
pregunta = "¿Cuál es el salario promedio de los empleados de la empresa?"
response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Contexto:\n{contexto_completo}\n\nPregunta: {pregunta}"
            }
        ],
        temperature=0
    )
sql_generado = response.choices[0].message.content

print(sql_generado)

No hay suficiente información para generar la consulta


The model correctly refrained from answering a question that lacked sufficient context.

In [ ]:
pregunta =  "¿Cuál es el total de dinero gastado por cada usuario en pedidos, considerando solo aquellos que tienen más de 2 órdenes?"
response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Contexto:\n{contexto_completo}\n\nPregunta: {pregunta}"
            }
        ],
        temperature=0
    )
sql_generado = response.choices[0].message.content

print(sql_generado)

SELECT 
    u.id AS user_id,
    u.first_name,
    u.last_name,
    SUM(o.total_amount) AS total_spent,
    COUNT(o.id) AS total_orders
FROM 
    users u
JOIN 
    orders o ON u.id = o.user_id
GROUP BY 
    u.id, u.first_name, u.last_name
HAVING 
    COUNT(o.id) > 2;


The model was also able to handle a more complex query using the document structure provided as context.

Although this baseline approach is functional, it may not be the most optimal solution. The model relies heavily on the LLM and its ability to interpret the full context correctly. While this works for a limited amount of metadata, scaling the solution would significantly increase the context length, making the system more expensive, slower, and potentially more prone to hallucinations.

With the baseline established, the next step is to improve the architecture by incorporating embeddings and Retrieval-Augmented Generation (RAG). To do this, the information will be split into two different components, as suggested during the course:

1. A document containing the table schema and table description, including metadata indicating which table it belongs to.
2. A key-value store containing the complete information for each table.

This structure allows the model to retrieve only the most relevant information before generating the SQL query, improving scalability, reducing token usage, and increasing the quality of the generated response.

## Final Model

In [ ]:
import os
import getpass
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver


# Build a key-value store where each table name acts as the key
# and the value contains all metadata associated with that table.
# This structure enables efficient retrieval of complete table information
# after identifying the most relevant table through semantic search.
table_store = {}

for table_name in df.columns:
    table_store[table_name] = {
        "table_name": table_name,
        "description": df.loc["description", table_name],
        "schema": df.loc["schema", table_name],
        "examples": df.loc["examples", table_name]
    }

In [ ]:
print(table_store["users"]["description"])

Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.


In [ ]:
#Metadata document
documents = []

for table_name in df.columns:
    description = df.loc["description", table_name]

    content = f"""
Table Name: {table_name}

Description:
{description}
""".strip()

    doc = Document(
        page_content=content,
        metadata={
            "table_name": table_name
        }
    )

    documents.append(doc)

In [ ]:
documents

[Document(metadata={'table_name': 'users'}, page_content='Table Name: users\n\nDescription:\nAlmacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.'),
 Document(metadata={'table_name': 'categories'}, page_content='Table Name: categories\n\nDescription:\nCategorías de productos para organizar el catálogo jerárquicamente.'),
 Document(metadata={'table_name': 'products'}, page_content='Table Name: products\n\nDescription:\nDetalles sobre artículos a la venta, precios, vinculación de categorías e inventario.'),
 Document(metadata={'table_name': 'orders'}, page_content='Table Name: orders\n\nDescription:\nDatos transaccionales de alto nivel para compras de clientes, estado del pedido y costo total.'),
 Document(metadata={'table_name': 'order_items'}, page_content='Table Name: order_items\n\nDescription:\nTabla de unión que detalla productos específicos comprados en un pedido, cantidades y precios cerrados.'),
 Document(metadata={'table_name': 'revi

 Defining the FAISS-based retrieval architecture for semantic search and contextual document retrieval

In [ ]:
import numpy as np
import faiss
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("OPENAI_TOKEN")
client = OpenAI(api_key=api_key)



In [ ]:
#Global Parameters:

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM=1536

#Model
GENERATION_MODEL="gpt-4.1-mini"
#Folder
data_folder='content/data/'
supported_ext='.pdf'
#RAG Retrieval
TOP_K=3
CHUNK_SIZE=500
CHUNK_OVERLAP=50



In [ ]:
SYSTEM_PROMPT = (
    "Eres un asistente experto en generación de consultas SQL.\n\n"

    "Tu tarea es generar consultas SQL basándote únicamente en el contexto proporcionado.\n"
    "El contexto incluye:\n"
    "- Nombre de las tablas\n"
    "- Descripción de las tablas\n"
    "- Schema (columnas y tipos)\n"
    "- Ejemplos de queries\n\n"

    "Reglas estrictas:\n"
    "1. SOLO puedes usar las tablas mencionadas en el contexto.\n"
    "2. SOLO puedes usar las columnas definidas en el schema.\n"
    "3. NO inventes tablas, columnas ni relaciones.\n"
    "4. Si la información no es suficiente para responder, di: 'No hay suficiente información para generar la consulta'.\n"
    "5. Genera únicamente consultas SQL válidas.\n"
    "6. NO incluyas explicaciones, comentarios ni texto adicional.\n"
    "7. NO uses lenguaje natural, SOLO SQL.\n"
    "8. Prefiere apoyarte en los patrones de los ejemplos proporcionados.\n\n"

    "Formato de salida:\n"
    "Devuelve únicamente la consulta SQL en texto plano.\n"
)

In [ ]:
#Embeddings function

def get_embedding(text: str, model: str = EMBEDDING_MODEL):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [ ]:
document_texts = [doc.page_content for doc in documents]
document_metadata = [doc.metadata for doc in documents]

embeddings_docs = []

for text in document_texts:
    emb = get_embedding(text)
    embeddings_docs.append(emb)

embeddings_docs = np.array(embeddings_docs, dtype="float32")

print("Shape embeddings:", embeddings_docs.shape)

Shape embeddings: (8, 1536)


In [ ]:
EMBEDDING_DIM = embeddings_docs.shape[1]

index = faiss.IndexFlatL2(EMBEDDING_DIM)
index.add(embeddings_docs)

print(f"Índice FAISS creado. Vectores indexados: {index.ntotal}")

Índice FAISS creado. Vectores indexados: 8


In [ ]:
### Core Functions for the Retrieval-Augmented Text-to-SQL Model


# Retrieve the most relevant tables using semantic similarity search
def retrieve_table_context(question: str, k: int = TOP_K, verbose: bool = True):
    """
    Retrieves the most relevant table documents for a given natural language question
    using FAISS vector similarity search.

    The function converts the user question into an embedding vector and searches
    the FAISS index to identify the top-k most semantically similar documents.

    Parameters
    ----------
    question : str
        Natural language question provided by the user.

    k : int, optional
        Number of top documents to retrieve.
        Default is defined by TOP_K.

    verbose : bool, optional
        If True, prints the retrieved documents and metadata for debugging
        and inspection purposes.

    Returns
    -------
    list
        List of retrieved LangChain Document objects ranked by semantic similarity.
    """

    # Generate embedding vector for the user question
    query_vec = np.array([get_embedding(question)], dtype="float32")

    # Perform similarity search against the FAISS index
    D, I = index.search(query_vec, min(k, index.ntotal))

    # Retrieve the corresponding documents from the document collection
    retrieved_docs = [documents[i] for i in I[0]]

    # Optional debugging output to inspect retrieved context
    if verbose:
        print("\n" + "=" * 60)
        print(f"Question: {question}")
        print(f"\nRetrieved Documents (top-{k}):")

        for j, doc in enumerate(retrieved_docs):
            print(f"\n[{j+1}] table_name = {doc.metadata['table_name']}")
            print(doc.page_content[:500])

    return retrieved_docs


# Retrieve complete metadata from the key-value store
def get_full_context_from_store(retrieved_docs, table_store):
    """
    Retrieves the complete metadata associated with each retrieved table.

    After semantic retrieval identifies the most relevant tables,
    this function uses the table name metadata to recover the full
    contextual information from the key-value store.

    Parameters
    ----------
    retrieved_docs : list
        List of retrieved LangChain Document objects.

    table_store : dict
        Dictionary containing the complete metadata for each table.

    Returns
    -------
    list
        List containing the full contextual information for each table.
    """

    full_contexts = []

    # Recover complete table information using the table name as key
    for doc in retrieved_docs:
        table_name = doc.metadata["table_name"]
        full_contexts.append(table_store[table_name])

    return full_contexts


# Build the final contextual prompt for the LLM
def build_prompt_context(full_contexts):
    """
    Builds the final structured context injected into the LLM prompt.

    The generated context includes:
    - Table names
    - Table descriptions
    - Schemas
    - Example SQL queries

    This contextual structure helps the model generate more accurate
    and grounded SQL queries while minimizing hallucinations.

    Parameters
    ----------
    full_contexts : list
        List containing complete metadata for the retrieved tables.

    Returns
    -------
    str
        Formatted prompt context ready to be injected into the LLM.
    """

    parts = []

    # Iterate through all retrieved table contexts
    for ctx in full_contexts:

        examples_text = ""

        # Format example SQL queries and descriptions
        for i, ex in enumerate(ctx["examples"], 1):
            examples_text += (
                f"\n{i}. Description: {ex.get('description')}\n"
                f"   SQL: {ex.get('query')}\n"
            )

        # Create a structured representation of the table
        table_text = f"""
Table Name: {ctx['table_name']}

Description:
{ctx['description']}

Schema:
{ctx['schema']}

Examples:
{examples_text}
""".strip()

        parts.append(table_text)

    # Join all retrieved table contexts into a single prompt section
    return "\n\n" + ("-" * 80 + "\n\n").join(parts)

In [ ]:
## Complete RAG Pipeline
def rag_query_sql(question: str, k: int = TOP_K, verbose: bool = True):
    """
    Complete Retrieval-Augmented Generation (RAG) pipeline for Text-to-SQL generation.

    Pipeline Steps
    --------------
    1. Generate an embedding from the user question.
    2. Retrieve the top-k most relevant documents using FAISS similarity search.
    3. Use document metadata to recover complete table information
       from the key-value store.
    4. Build the final contextual prompt.
    5. Generate the SQL query using the LLM.

    Parameters
    ----------
    question : str
        Natural language question provided by the user.

    k : int, optional
        Number of documents to retrieve from the vector database.
        Default is defined by TOP_K.

    verbose : bool, optional
        If True, prints debugging information including:
        - Retrieved context
        - Final prompt preview
        - Generated SQL query
        - Token usage

    Returns
    -------
    str
        SQL query generated by the language model.
    """

    # ------------------------------------------------------------
    # Step 1: Semantic Retrieval
    # Retrieve the most relevant table documents using FAISS
    # similarity search over embedding vectors
    # ------------------------------------------------------------
    retrieved_docs = retrieve_table_context(
        question,
        k=k,
        verbose=verbose
    )

    # ------------------------------------------------------------
    # Step 2: Metadata Lookup
    # Recover the complete contextual information associated
    # with each retrieved table from the key-value store
    # ------------------------------------------------------------
    full_contexts = get_full_context_from_store(
        retrieved_docs,
        table_store
    )

    # ------------------------------------------------------------
    # Step 3: Prompt Context Construction
    # Build the final structured context that will be injected
    # into the LLM prompt
    # ------------------------------------------------------------
    contexto_rag = build_prompt_context(full_contexts)

    # Final user prompt combining retrieved context and question
    prompt_usuario = (
        f"Contexto:\n{contexto_rag}\n\n"
        f"Pregunta: {question}"
    )

    # ------------------------------------------------------------
    # Step 4: SQL Generation
    # Generate the SQL query using the selected LLM
    # ------------------------------------------------------------
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt_usuario
            }
        ],
        temperature=0
    )

    # Extract generated SQL response
    sql_output = response.choices[0].message.content.strip()

    # ------------------------------------------------------------
    # Optional debugging and inspection output
    # ------------------------------------------------------------
    if verbose:

        print("\n" + "=" * 60)
        print("FINAL CONTEXT SENT TO THE MODEL:")
        print(contexto_rag[:3000])  # Context preview

        print("\n" + "=" * 60)
        print(f"MODEL RESPONSE ({GENERATION_MODEL}):")
        print(sql_output)

        print(f"\nTotal Tokens Used: {response.usage.total_tokens}")

    return sql_output

In [ ]:
#Question 1
sql_1 = rag_query_sql("Dame todos los usuarios registrados con su correo electrónico.")


Pregunta: Dame todos los usuarios registrados con su correo electrónico.

Documentos recuperados (top-3):

[1] table_name = users
Table Name: users

Description:
Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.

[2] table_name = shipping_details
Table Name: shipping_details

Description:
Información logística, números de seguimiento y direcciones de entrega vinculadas a un pedido.

[3] table_name = reviews
Table Name: reviews

Description:
Calificaciones de clientes y comentarios escritos para productos específicos.

CONTEXTO FINAL ENVIADO AL MODELO:


Table Name: users

Description:
Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.

Schema:
id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP

Examples:

1. Description: Buscar un usuario por correo electrónico.
   SQL: None

2. Descr

In [ ]:
# Question without sufficient context:
sql_2 = rag_query_sql("¿Cuál es el salario promedio de los empleados de la empresa?")


Pregunta: ¿Cuál es el salario promedio de los empleados de la empresa?

Documentos recuperados (top-3):

[1] table_name = promotions
Table Name: promotions

Description:
Códigos de descuento, porcentajes de oferta y rangos de fechas válidos para promociones de la tienda.

[2] table_name = users
Table Name: users

Description:
Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.

[3] table_name = orders
Table Name: orders

Description:
Datos transaccionales de alto nivel para compras de clientes, estado del pedido y costo total.

CONTEXTO FINAL ENVIADO AL MODELO:


Table Name: promotions

Description:
Códigos de descuento, porcentajes de oferta y rangos de fechas válidos para promociones de la tienda.

Schema:
id INT PRIMARY KEY, code VARCHAR(50) UNIQUE, discount_percentage DECIMAL(5, 2), start_date DATE, end_date DATE, is_active BOOLEAN

Examples:

1. Description: Verificar si un código de descuento está activo y es válido hoy.
   SQL: None

In [ ]:
## More complex question:
sql_3 = rag_query_sql("¿Cuál es el total de dinero gastado por cada usuario en pedidos, considerando solo aquellos que tienen más de 2 órdenes?")



Pregunta: ¿Cuál es el total de dinero gastado por cada usuario en pedidos, considerando solo aquellos que tienen más de 2 órdenes?

Documentos recuperados (top-3):

[1] table_name = orders
Table Name: orders

Description:
Datos transaccionales de alto nivel para compras de clientes, estado del pedido y costo total.

[2] table_name = order_items
Table Name: order_items

Description:
Tabla de unión que detalla productos específicos comprados en un pedido, cantidades y precios cerrados.

[3] table_name = shipping_details
Table Name: shipping_details

Description:
Información logística, números de seguimiento y direcciones de entrega vinculadas a un pedido.

CONTEXTO FINAL ENVIADO AL MODELO:


Table Name: orders

Description:
Datos transaccionales de alto nivel para compras de clientes, estado del pedido y costo total.

Schema:
id INT PRIMARY KEY, user_id INT, order_date TIMESTAMP, status VARCHAR(50), total_amount DECIMAL(10, 2)

Examples:

1. Description: Obtener pedidos pendientes para 

This model is more efficient than the baseline because it does not rely exclusively on the LLM to determine the final answer. Instead, it uses embeddings and FAISS-based semantic retrieval to identify and send only the most relevant context to the model.

By reducing the amount of irrelevant information included in the prompt, this approach lowers noise, improves contextual precision, and reduces the API cost associated with each query.
